# **Análise baseada especifica baseada na região caririense**

> Totais de Cidades inclusas na análise: 28

Abaiara, Altaneira, Antonina do Norte, Araripe, Assaré, Aurora, Barbalha, Barro, Brejo Santo, Campos Sales, Caririaçu, Crato, Farias Brito, Granjeiro, Jardim, Jati, Juazeiro do Norte, Mauriti, Milagres, Missão Velha, Nova Olinda, Penaforte, Porteiras, Potengi, Salitre, Santana do Cariri, Tarrafas, Várzea Alegre.

#### _Importando as dependencias_

In [4]:
import pandas as pd                          # Manipulação dos dados
import numpy as np                           # Calculos Matmáticos
import matplotlib.pyplot as plt              # Geração de Gráficos
import seaborn as sns                        # Graficos automaticos, baseado no matplotlib
import statistics as sta                     # Biblioteca padrão do py para estatistica
from scipy import stats                      # Estastistica avançada
import os                                    # SO

### **Limpeza e Preparação**

In [7]:
cito_mamo = pd.read_csv('../../Datasets/cito_e_mamo_cariri.csv')
print(cito_mamo.columns.tolist())
cito_mamo.head(10)

['estado', 'municipio', 'valor_indicador', 'ano_mes', 'data_atualizacao', 'tipo_exame']


,estado,municipio,valor_indicador,ano_mes,data_atualizacao,tipo_exame
0,CE,Brejo Santo,748.0,201612,2026-03-17 01:25:03,Citopatologico
1,CE,Crato,16742.0,201612,2026-03-17 01:25:03,Citopatologico
2,CE,Juazeiro do Norte,4176.0,201612,2026-03-17 01:25:03,Citopatologico
3,CE,Barbalha,1.0,201712,2026-03-17 01:22:16,Citopatologico
4,CE,Brejo Santo,4251.0,201712,2026-03-17 01:22:16,Citopatologico
5,CE,Crato,19052.0,201712,2026-03-17 01:22:16,Citopatologico
6,CE,Juazeiro do Norte,4649.0,201712,2026-03-17 01:22:16,Citopatologico
7,CE,Barbalha,33.0,201812,2026-03-17 01:20:07,Citopatologico
8,CE,Brejo Santo,5414.0,201812,2026-03-17 01:20:07,Citopatologico
9,CE,Crato,13976.0,201812,2026-03-17 01:20:07,Citopatologico


#### _Visão geral do dataset cito_e_mamo.csv_

Observação e primeira visão dos dados e tipos de dados:
- Tipo de valor, valores nulos, valores unicos (quantidade de valores diferentes)... 

Estastica: 
- Média, Mediana, Mínimo, Máximo e Desvio padrão para a coluna do valor_indicador

> Foi utilizado a métrica IQR por não ser sensível a Outlier como a métrica tradicional


In [37]:
# Gerando documentação das colunas
print("# DOCUMENTAÇÃO DO DATASET - EXAMES CITOPATOLÓGICOS\n")
print("## Visão Geral")
print(f"- **Total de registros**: {len(cito_mamo):,}")
print(f"- **Total de colunas**: {len(cito_mamo.columns)}")
print(f"- **Última atualização**: 24 de novembro de 2025\n")

print("## Descrição Detalhada das Colunas\n")

for i, column in enumerate(cito_mamo.columns, 1):
    print(f"### {i}. {column}")
    print(f"**Tipo de dado**: {cito_mamo[column].dtype}")
    print(f"**Valores únicos**: {cito_mamo[column].nunique():,}")
    print(f"**Valores nulos**: {cito_mamo[column].isnull().sum():,} ({cito_mamo[column].isnull().sum()/len(cito_mamo)*100:.1f}%)")

    # Para colunas numéricas
    if cito_mamo[column].dtype in ['float64']:
        print("**Estatísticas**:")
        print(f"  - Média: {cito_mamo[column].mean():,.2f}")
        print(f"  - Mediana: {cito_mamo[column].median():,.2f}")
        print(f"  - Mínimo: {cito_mamo[column].min():,}")
        print(f"  - Máximo: {cito_mamo[column].max():,}")

        q1 = cito_mamo[column].quantile(0.25)
        q3 = cito_mamo[column].quantile(0.75)
        iqr = q3 - q1
        std_robust = iqr / 1.349  # Fórmula robusta: σ ≈ IQR / 1.349 (para distribuição normal)
        print(f"  - Desvio padrão: {std_robust:,.2f}")
        print(f"  - IQR (Intervalo Interquartílico): {iqr:,.2f}")
    else:
        unique_values = cito_mamo[column].dropna().unique()
        if len(unique_values) <= 10:
            print(f"**Valores únicos**: {', '.join(str(v) for v in unique_values)}")
        else:
            print(f"**Primeiros valores únicos**: {', '.join(str(v) for v in unique_values[:10])}...")

    print()

print("---")
print("*Documentação gerada automaticamente em", pd.Timestamp.now().strftime("%d/%m/%Y %H:%M"))

# DOCUMENTAÇÃO DO DATASET - EXAMES CITOPATOLÓGICOS

## Visão Geral
- **Total de registros**: 114
- **Total de colunas**: 6
- **Última atualização**: 24 de novembro de 2025

## Descrição Detalhada das Colunas

### 1. estado
**Tipo de dado**: str
**Valores únicos**: 1
**Valores nulos**: 0 (0.0%)
**Valores únicos**: CE

### 2. municipio
**Tipo de dado**: str
**Valores únicos**: 17
**Valores nulos**: 0 (0.0%)
**Primeiros valores únicos**: Brejo Santo, Crato, Juazeiro do Norte, Barbalha, Farias Brito, Missão Velha, Campos Sales, Várzea Alegre, Abaiara, Mauriti...

### 3. valor_indicador
**Tipo de dado**: float64
**Valores únicos**: 108
**Valores nulos**: 0 (0.0%)
**Estatísticas**:
  - Média: 3,345.48
  - Mediana: 1,255.00
  - Mínimo: 1.0
  - Máximo: 22,309.0
  - Desvio padrão: 2,825.24
  - IQR (Intervalo Interquartílico): 3,811.25

### 4. ano_mes
**Tipo de dado**: int64
**Valores únicos**: 10
**Valores nulos**: 0 (0.0%)
**Valores únicos**: 201612, 201712, 201812, 201912, 202012, 202112, 202

#### _Desvio padrão e seus detalhamentos_

Nesse ponto foi utilizado a métrica IQR, para facilitar a captura de outliers

MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO <br>
O que é:

> O IQR divide os dados em 4 partes iguais (quartis). <br>
> Valores que saem dos limites IQR são considerados outliers.

In [33]:
print("MÉTRICAS DE PADRÃO E DETECÇÃO DE ANOMALIAS")

if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador']
    
    print(f"\n MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO")
    print("-" * 100)
    
    q1 = valor_indicador.quantile(0.25)
    q2 = valor_indicador.quantile(0.50)
    q3 = valor_indicador.quantile(0.75)
    iqr = q3 - q1
    
    print(f" VALORES DO DATASET:")
    print(f"   • Q1 (25º percentil): {q1:,.2f}")
    print(f"   • Q2 (50º percentil/Mediana): {q2:,.2f}")
    print(f"   • Q3 (75º percentil): {q3:,.2f}")
    print(f"   • IQR (Q3 - Q1): {iqr:,.2f}")
    
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    
    print(f"\n QUANDO FOGE DO PADRÃO (método IQR - 1.5x):")
    print(f"   • Limite inferior: Q1 - 1.5 × IQR = {q1:,.2f} - 1.5 × {iqr:,.2f} = {limite_inferior:,.2f}")
    print(f"   • Limite superior: Q3 + 1.5 × IQR = {q3:,.2f} + 1.5 × {iqr:,.2f} = {limite_superior:,.2f}")
    print(f"   • Outliers: valores < {limite_inferior:,.2f} ou > {limite_superior:,.2f}")
    
    outliers_iqr = valor_indicador[(valor_indicador < limite_inferior) | (valor_indicador > limite_superior)]
    
    print(f"\n NO DATASET:")
    print(f"   • Total de outliers (IQR): {len(outliers_iqr)} ({len(outliers_iqr)/len(valor_indicador)*100:.1f}%)")
    print(f"   • Valores mínimo/máximo reais: {valor_indicador.min():,.2f} / {valor_indicador.max():,.2f}")
    
    print(f"   • Valores fora do intervalo [{limite_inferior:,.2f}, {limite_superior:,.2f}] são anomalias")

else:
    print(" Coluna 'valor_indicador' não encontrada no dataset!")

MÉTRICAS DE PADRÃO E DETECÇÃO DE ANOMALIAS

 MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO
----------------------------------------------------------------------------------------------------
 VALORES DO DATASET:
   • Q1 (25º percentil): 155.00
   • Q2 (50º percentil/Mediana): 1,255.00
   • Q3 (75º percentil): 3,966.25
   • IQR (Q3 - Q1): 3,811.25

 QUANDO FOGE DO PADRÃO (método IQR - 1.5x):
   • Limite inferior: Q1 - 1.5 × IQR = 155.00 - 1.5 × 3,811.25 = -5,561.88
   • Limite superior: Q3 + 1.5 × IQR = 3,966.25 + 1.5 × 3,811.25 = 9,683.12
   • Outliers: valores < -5,561.88 ou > 9,683.12

 NO DATASET:
   • Total de outliers (IQR): 13 (11.4%)
   • Valores mínimo/máximo reais: 1.00 / 22,309.00
   • Valores fora do intervalo [-5,561.88, 9,683.12] são anomalias


Mostrando quem faz parte desse desvio...

In [47]:
print("ANOMALIAS DETECTADAS: Linhas que Fogem do Padrão (Outliers via IQR)")

if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador']

    q1 = valor_indicador.quantile(0.25)
    q3 = valor_indicador.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers_df = cito_mamo[(valor_indicador < limite_inferior) | (valor_indicador > limite_superior)].copy()

    outliers_df['tipo_outlier'] = outliers_df['valor_indicador'].apply(
        lambda x: 'Outlier Inferior' if x < limite_inferior else 'Outlier Superior'
    )

    mediana = valor_indicador.median()
    outliers_df['desvio_mediana'] = outliers_df['valor_indicador'] - mediana

    print(f"\n RESUMO DOS OUTLIERS:")
    print(f"-" * 50)
    print(f"Total de registros no dataset: {len(cito_mamo)}")
    print(f"Total de outliers detectados: {len(outliers_df)} ({len(outliers_df)/len(cito_mamo)*100:.1f}%)")
    print(f"Limites IQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")
    print(f"Mediana do dataset: {mediana:,.2f}\n")

    if len(outliers_df) > 0:
        print(f"TABELA COMPLETA DOS OUTLIERS:")
        print(f"-" * 120)

        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        cols_importantes = ['valor_indicador', 'tipo_outlier', 'desvio_mediana']
        outras_cols = [col for col in outliers_df.columns if col not in cols_importantes]
        ordem_cols = cols_importantes + outras_cols

        print(outliers_df[ordem_cols].to_string(index=True))

        print(f"\n" + "-" * 120)
        print(f"INTERPRETAÇÃO:")
        print(f"- **Outlier Inferior**: Valores muito baixos (abaixo de {limite_inferior:,.2f})")
        print(f"- **Outlier Superior**: Valores muito altos (acima de {limite_superior:,.2f})")
        print(f"- **Desvio da Mediana**: Quanto o valor se afasta da mediana ({mediana:,.2f})")
        print(f"- Valores positivos = acima da mediana | Valores negativos = abaixo da mediana")
        print(f"- Quanto maior o desvio absoluto, mais extrema é a anomalia\n")

        print(f"-" * 50)
        print(f"Valor indicador - Mínimo: {outliers_df['valor_indicador'].min():,.2f}")
        print(f"Valor indicador - Máximo: {outliers_df['valor_indicador'].max():,.2f}")
        print(f"Valor indicador - Média: {outliers_df['valor_indicador'].mean():,.2f}")
        print(f"Desvio da mediana - Médio: {outliers_df['desvio_mediana'].mean():,.2f}")
        print(f"Outliers inferiores: {len(outliers_df[outliers_df['tipo_outlier'] == 'Outlier Inferior'])}")
        print(f"Outliers superiores: {len(outliers_df[outliers_df['tipo_outlier'] == 'Outlier Superior'])}")

    else:
        print("Todos os valores estão dentro dos limites considerados normais pelo método IQR.")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")

ANOMALIAS DETECTADAS: Linhas que Fogem do Padrão (Outliers via IQR)

 RESUMO DOS OUTLIERS:
--------------------------------------------------
Total de registros no dataset: 114
Total de outliers detectados: 13 (11.4%)
Limites IQR: [-5,561.88, 9,683.12]
Mediana do dataset: 1,255.00

TABELA COMPLETA DOS OUTLIERS:
------------------------------------------------------------------------------------------------------------------------
    valor_indicador      tipo_outlier  desvio_mediana estado          municipio  ano_mes     data_atualizacao      tipo_exame
1           16742.0  Outlier Superior         15487.0     CE              Crato   201612  2026-03-17 01:25:03  Citopatologico
5           19052.0  Outlier Superior         17797.0     CE              Crato   201712  2026-03-17 01:22:16  Citopatologico
9           13976.0  Outlier Superior         12721.0     CE              Crato   201812  2026-03-17 01:20:07  Citopatologico
10           9735.0  Outlier Superior          8480.0     CE  

In [ ]:
# Filtrando linhas onde valor_indicador foge do intervalo [-5,561.88, 9,683.12]
print("\n" + "=" * 120)
print("FILTRO ESPECÍFICO: Linhas Fora do Intervalo [-5,561.88, 9,683.12]")
print("=" * 120 + "\n")

if 'valor_indicador' in cito_mamo.columns:
    # Definir o intervalo especificado
    limite_inferior = -5561.88  # Convertendo -5,561.88 para formato numérico
    limite_superior = 9683.12   # Convertendo 9,683.12 para formato numérico

    # Filtrar registros fora do intervalo
    filtro_df = cito_mamo[(cito_mamo['valor_indicador'] < limite_inferior) |
                          (cito_mamo['valor_indicador'] > limite_superior)].copy()

    # Adicionar coluna de classificação
    filtro_df['status_intervalo'] = filtro_df['valor_indicador'].apply(
        lambda x: 'Abaixo do Limite' if x < limite_inferior else 'Acima do Limite'
    )

    # Adicionar coluna de desvio dos limites
    filtro_df['desvio_limite_inferior'] = filtro_df['valor_indicador'] - limite_inferior
    filtro_df['desvio_limite_superior'] = filtro_df['valor_indicador'] - limite_superior

    print(f"📊 RESUMO DO FILTRO:")
    print(f"-" * 50)
    print(f"Intervalo especificado: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")
    print(f"Total de registros no dataset: {len(cito_mamo)}")
    print(f"Registros fora do intervalo: {len(filtro_df)} ({len(filtro_df)/len(cito_mamo)*100:.1f}%)")
    print(f"Registros dentro do intervalo: {len(cito_mamo) - len(filtro_df)} ({(len(cito_mamo) - len(filtro_df))/len(cito_mamo)*100:.1f}%)")
    print()

    if len(filtro_df) > 0:
        print(f"🔴 TABELA: REGISTROS FORA DO INTERVALO")
        print(f"-" * 120)

        # Configurar pandas para exibir todas as colunas
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        # Reordenar colunas para destacar as importantes
        cols_importantes = ['valor_indicador', 'status_intervalo', 'desvio_limite_inferior', 'desvio_limite_superior']
        outras_cols = [col for col in filtro_df.columns if col not in cols_importantes]
        ordem_cols = cols_importantes + outras_cols

        # Exibir a tabela
        print(filtro_df[ordem_cols].to_string(index=True))

        print(f"\n" + "-" * 120)
        print(f"💡 INTERPRETAÇÃO:")
        print(f"- **Abaixo do Limite**: Valores < {limite_inferior:,.2f}")
        print(f"- **Acima do Limite**: Valores > {limite_superior:,.2f}")
        print(f"- **Desvio Limite Inferior**: Diferença em relação ao limite mínimo")
        print(f"- **Desvio Limite Superior**: Diferença em relação ao limite máximo")
        print(f"- Valores negativos no desvio inferior = muito abaixo | Valores positivos no desvio superior = muito acima")

        # Estatísticas dos registros filtrados
        print(f"\n📈 ESTATÍSTICAS DOS REGISTROS FILTRADOS:")
        print(f"-" * 50)
        print(f"Valor indicador - Mínimo: {filtro_df['valor_indicador'].min():,.2f}")
        print(f"Valor indicador - Máximo: {filtro_df['valor_indicador'].max():,.2f}")
        print(f"Valor indicador - Média: {filtro_df['valor_indicador'].mean():,.2f}")
        print(f"Registros abaixo do limite: {len(filtro_df[filtro_df['status_intervalo'] == 'Abaixo do Limite'])}")
        print(f"Registros acima do limite: {len(filtro_df[filtro_df['status_intervalo'] == 'Acima do Limite'])}")

    else:
        print("✅ NENHUM REGISTRO FORA DO INTERVALO!")
        print(f"Todos os valores estão dentro do intervalo [{limite_inferior:,.2f}, {limite_superior:,.2f}].")

else:
    print("⚠️  Coluna 'valor_indicador' não encontrada no dataset!")

### **Análise Exploratória (EDA- Exploratory Data Analysis)**

Analise realizada continuada com base no Notebook5